# 02 — Hospital building data EDA

Explore building characteristics, missingness, and data issues in HCAI’s Hospital Building Data. Records describe buildings and related campus structures; a facility can have many records. The source filename indicates a September 3, 2026 snapshot.

Height, stories, design code, and completion year complement the seismic assessments in notebook 01. Building-code year identifies a code edition, not the building’s age. See the [data dictionary](../docs/HOSPITAL_BUILDING_DATA.md#detailed-data-dictionary) for definitions and [data sources](../docs/DATA_SOURCES.md#hospital-building-data) for provenance.

All records remain in this analysis; raw files are unchanged.


In [1]:
import csv
import hashlib
import io
import math
import re
import statistics
from collections import Counter, defaultdict
from pathlib import Path
from IPython.display import Markdown, display

ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "data/raw").is_dir()),
    None,
)
if ROOT is None:
    raise RuntimeError("Open this notebook from the repository root or notebooks directory.")

SOURCE = ROOT / "data/raw/hospital-building-data-.csv"

from urllib.parse import urlparse


In [2]:
IN_SERVICE = "OSHPD 1-In Service"
SNAPSHOT_YEAR = 2026  # From the recorded source download date.
FEATURES = ("Height (ft)", "Stories", "Building Code Year", "Year Completed")
NUMERIC_COLUMNS = (*FEATURES, "Latitude", "Longitude", "Count")
TYPES = {
    "County Code": "Text/category",
    "Perm ID": "Text identifier",
    "Facility Name": "Text",
    "City": "Text/category",
    "Building Nbr": "Text identifier",
    "Building Name": "Text",
    "Building Status": "Category",
    "SPC Rating *": "Category; retain suffixes",
    "Building URL": "URL/text",
    "Height (ft)": "Nullable numeric feet",
    "Stories": "Nullable integer",
    "Building Code": "Category; preserve edition and family",
    "Building Code Year": "Nullable integer year",
    "Year Completed": "Nullable integer year",
    "AB 1882 Notice": "Text/category with optional notice",
    "Latitude": "Numeric coordinate",
    "Longitude": "Numeric coordinate",
    "Count": "Integer count helper",
}


In [3]:
def read_source(path):
    raw = path.read_bytes()
    reader = csv.reader(io.StringIO(raw.decode("cp1252"), newline=""))
    headers = next(reader, [])
    normalized = [header.strip() for header in headers]
    if len(set(normalized)) != len(normalized) or set(normalized) != set(TYPES):
        raise ValueError("Unexpected columns; review the source schema before profiling.")
    rows = []
    for line, values in enumerate(reader, start=2):
        if len(values) != len(headers):
            raise ValueError(f"CSV record {line} has {len(values)} fields; expected {len(headers)}.")
        rows.append(dict(zip(normalized, values)))
    if not rows:
        raise ValueError("The input CSV has no observations.")
    return raw, headers, rows


In [4]:
def number(value):
    try:
        parsed = float(value)
    except ValueError:
        return None
    return parsed if math.isfinite(parsed) else None

def blank(value):
    return not value.strip()

def inferred_type(values):
    nonblank = [v for v in values if not blank(v)]
    if not nonblank:
        return "All blank"
    if all(re.fullmatch(r"[+-]?\d+", v.strip()) for v in nonblank):
        return "Integer-like"
    if all(number(v) is not None for v in nonblank):
        return "Numeric-like"
    return "Text/mixed labels"


In [5]:
def numeric_summary(rows, column):
    supplied = [r[column] for r in rows if not blank(r[column])]
    parsed = [number(v) for v in supplied]
    values = [v for v in parsed if v is not None]
    stats = [None] * 6
    if values:
        quartiles = statistics.quantiles(values, n=4, method="inclusive") if len(values) > 1 else values * 3
        stats = [min(values), quartiles[0], statistics.median(values), quartiles[2], max(values), statistics.mean(values)]
    return {
        "values": values,
        "invalid": sum(v is None for v in parsed),
        "stats": stats,
        "zero": sum(v == 0 for v in values),
        "negative": sum(v < 0 for v in values),
        "fractional": sum(not v.is_integer() for v in values),
    }


In [6]:
def table(headers, rows):
    def escape(value):
        return str(value).replace("|", "\\|").replace("\n", " ")
    lines = ["| " + " | ".join(map(escape, headers)) + " |", "| " + " | ".join("---" for _ in headers) + " |"]
    lines.extend("| " + " | ".join(map(escape, row)) + " |" for row in rows)
    return "\n".join(lines)

def rate(count, total):
    return f"{100 * count / total:.1f}%" if total else "—"

def fmt(value):
    return "—" if value is None else f"{value:,.4f}".rstrip("0").rstrip(".")


In [7]:
def display_sections(sections):
    display(Markdown("\n\n".join(sections)))


In [8]:
raw, headers, rows = read_source(SOURCE.resolve())
input_sha256 = hashlib.sha256(raw).hexdigest()
n = len(rows)
counts = {c: Counter(r[c] for r in rows) for c in TYPES}
numeric = {c: numeric_summary(rows, c) for c in NUMERIC_COLUMNS}
in_service = [r for r in rows if r["Building Status"] == IN_SERVICE]
facilities = defaultdict(list)
names = defaultdict(set)
coordinates = defaultdict(set)
for r in rows:
    facilities[r["Perm ID"]].append(r)
    names[r["Facility Name"]].add(r["Perm ID"])
    coordinates[(r["Latitude"], r["Longitude"])].add(r["Perm ID"])
city_conflicts = {
    key: Counter(r["City"] for r in group)
    for key, group in facilities.items()
    if len({r["City"] for r in group}) > 1
}
duplicate_rows = n - len({tuple(r[c] for c in TYPES) for r in rows})
duplicate_keys = n - len({(r["Perm ID"], r["Building Nbr"]) for r in rows})
complete_features = [r for r in rows if all(number(r[c]) is not None for c in FEATURES)]
print(f"Loaded {n:,} records and {len(headers)} columns.")
print(f"Input SHA-256: {input_sha256}")


Loaded 4,690 records and 18 columns.
Input SHA-256: 57731b735091940e6445ed85cf80f3bca27b366be0554d28378e42eb71b679ab


## Snapshot overview

The exact status `OSHPD 1-In Service` is a comparison subset, not the final modeling population.


In [9]:
display_sections([
    f"- {n:,} records, {len(headers)} columns, {len(facilities):,} facility IDs, and {len(counts['County Code'])} counties.\n"
    f"- {duplicate_rows:,} exact duplicate rows and {duplicate_keys:,} duplicate facility/building key pairs.\n"
    f"- {len(in_service):,} records ({rate(len(in_service), n)}) have status `{IN_SERVICE}`.\n"
    f"- {len(complete_features):,} records ({rate(len(complete_features), n)}) have numeric values in all four building features; this includes zero values.\n"
    f"- {sum(blank(r['Height (ft)']) for r in rows):,} heights and {sum(blank(r['Year Completed']) for r in rows):,} completion years are blank."
])


- 4,690 records, 18 columns, 423 facility IDs, and 56 counties.
- 0 exact duplicate rows and 0 duplicate facility/building key pairs.
- 3,183 records (67.9%) have status `OSHPD 1-In Service`.
- 2,289 records (48.8%) have numeric values in all four building features; this includes zero values.
- 2,361 heights and 889 completion years are blank.

## Column types and missingness

Cells are loaded as text. Distinct counts exclude blanks; literal `N/A`, `NYA`, and `Unknown` are counted separately. IDs stay text, even when they look numeric. Zero is a supplied value, not a blank.


In [10]:
schema = []
for c in TYPES:
    values = [r[c] for r in rows]
    empty = sum(blank(v) for v in values)
    schema.append([
        c, inferred_type(values), TYPES[c], len({v for v in values if not blank(v)}),
        empty, rate(empty, n), counts[c]["N/A"], counts[c]["NYA"], counts[c]["Unknown"],
    ])
display_sections([table(
    ["Column", "Observed form", "Recommended type", "Distinct nonblank", "Blank", "Blank %", "N/A", "NYA", "Unknown"],
    schema,
)])


| Column | Observed form | Recommended type | Distinct nonblank | Blank | Blank % | N/A | NYA | Unknown |
| --- | --- | --- | --- | --- | --- | --- | --- | --- |
| County Code | Text/mixed labels | Text/category | 56 | 0 | 0.0% | 0 | 0 | 0 |
| Perm ID | Integer-like | Text identifier | 423 | 0 | 0.0% | 0 | 0 | 0 |
| Facility Name | Text/mixed labels | Text | 416 | 0 | 0.0% | 0 | 0 | 0 |
| City | Text/mixed labels | Text/category | 251 | 0 | 0.0% | 0 | 0 | 0 |
| Building Nbr | Text/mixed labels | Text identifier | 4690 | 0 | 0.0% | 0 | 0 | 0 |
| Building Name | Text/mixed labels | Text | 2856 | 0 | 0.0% | 0 | 0 | 0 |
| Building Status | Text/mixed labels | Category | 28 | 0 | 0.0% | 0 | 0 | 0 |
| SPC Rating * | Text/mixed labels | Category; retain suffixes | 12 | 0 | 0.0% | 1141 | 0 | 0 |
| Building URL | Text/mixed labels | URL/text | 4690 | 0 | 0.0% | 0 | 0 | 0 |
| Height (ft) | Numeric-like | Nullable numeric feet | 817 | 2361 | 50.3% | 0 | 0 | 0 |
| Stories | Integer-like | Nullable integer | 16 | 1074 | 22.9% | 0 | 0 | 0 |
| Building Code | Text/mixed labels | Category; preserve edition and family | 67 | 0 | 0.0% | 0 | 0 | 80 |
| Building Code Year | Integer-like | Nullable integer year | 45 | 80 | 1.7% | 0 | 0 | 0 |
| Year Completed | Integer-like | Nullable integer year | 104 | 889 | 19.0% | 0 | 0 | 0 |
| AB 1882 Notice | Text/mixed labels | Text/category with optional notice | 2 | 3936 | 83.9% | 0 | 0 | 0 |
| Latitude | Numeric-like | Numeric coordinate | 422 | 0 | 0.0% | 0 | 0 | 0 |
| Longitude | Numeric-like | Numeric coordinate | 422 | 0 | 0.0% | 0 | 0 | 0 |
| Count | Integer-like | Integer count helper | 1 | 0 | 0.0% | 0 | 0 | 0 |

## Numeric distributions

Summaries use finite, nonblank numbers and retain zeros. Quartiles use linear interpolation. Coordinates are repeated for buildings at the same facility, so these summaries are weighted by building records.


In [11]:
display_sections([table(
    ["Column", "Valid n", "Invalid/nonfinite", "Min", "Q1", "Median", "Q3", "Max", "Mean", "Zeros"],
    [[c, len(v["values"]), v["invalid"], *map(fmt, v["stats"]), v["zero"]] for c, v in numeric.items()],
)])


| Column | Valid n | Invalid/nonfinite | Min | Q1 | Median | Q3 | Max | Mean | Zeros |
| --- | --- | --- | --- | --- | --- | --- | --- | --- | --- |
| Height (ft) | 2329 | 0 | 0 | 13 | 16.67 | 33.5 | 262.5 | 29.015 | 22 |
| Stories | 3616 | 0 | 0 | 1 | 1 | 2 | 17 | 1.9087 | 48 |
| Building Code Year | 4610 | 0 | 1,927 | 1,973 | 1,992 | 2,010 | 2,025 | 1,991.4544 | 0 |
| Year Completed | 3801 | 0 | 1,902 | 1,975 | 1,993 | 2,009 | 2,026 | 1,990.8203 | 0 |
| Latitude | 4690 | 0 | 32.6189 | 33.8997 | 34.363 | 37.7061 | 41.7745 | 35.6637 | 0 |
| Longitude | 4690 | 0 | -124.194 | -121.434 | -118.4875 | -117.8907 | -114.5951 | -119.4325 | 0 |
| Count | 4690 | 0 | 1 | 1 | 1 | 1 | 1 | 1 | 0 |

## Building status and feature availability

Feature availability can differ by structure and use. Percentages below use each status’s record count as the denominator and count finite numeric values, including zeros.


In [12]:
status_rows = []
for status, count in counts["Building Status"].most_common():
    group = [r for r in rows if r["Building Status"] == status]
    available = [sum(number(r[c]) is not None for r in group) for c in FEATURES]
    status_rows.append([status, count, rate(count, n), *(rate(v, count) for v in available)])
display_sections([table(
    ["Status", "Records", "% of all", "Height available", "Stories available", "Code year available", "Completion year available"],
    status_rows,
)])


| Status | Records | % of all | Height available | Stories available | Code year available | Completion year available |
| --- | --- | --- | --- | --- | --- | --- |
| OSHPD 1-In Service | 3183 | 67.9% | 67.0% | 97.3% | 99.3% | 99.6% |
| OSHPD 1-Proposed | 512 | 10.9% | 0.2% | 16.2% | 99.6% | 0.2% |
| OSHPD 1-Under Construction | 304 | 6.5% | 2.6% | 23.0% | 100.0% | 3.0% |
| OSHPD 1-Not an Independent Building | 169 | 3.6% | 20.1% | 79.9% | 98.8% | 97.6% |
| OSHPD 1-Equipment Yard | 156 | 3.3% | 12.8% | 16.7% | 95.5% | 95.5% |
| OSHPD 1-Tanks | 86 | 1.8% | 4.7% | 7.0% | 84.9% | 82.6% |
| OSHPD 1R-No Gen Acute Care - OSHPD Bldg | 79 | 1.7% | 67.1% | 91.1% | 82.3% | 94.9% |
| OSHPD 1-Tunnels | 61 | 1.3% | 32.8% | 57.4% | 96.7% | 96.7% |
| OSHPD 1-Not a Building Structure | 45 | 1.0% | 82.2% | 93.3% | 97.8% | 97.8% |
| OSHPD 2-Skilled Nursing Only | 29 | 0.6% | 31.0% | 48.3% | 89.7% | 89.7% |
| OSHPD 5-Acute Psych Only | 9 | 0.2% | 11.1% | 55.6% | 77.8% | 88.9% |
| OSHPD 5-Under Construction | 9 | 0.2% | 0.0% | 0.0% | 100.0% | 0.0% |
| OSHPD 3 - Local-Clinic - Local Jurisdiction | 8 | 0.2% | 0.0% | 37.5% | 75.0% | 12.5% |
| OSHPD 3 - Local-Outpatient Only | 8 | 0.2% | 0.0% | 62.5% | 62.5% | 87.5% |
| OSHPD 3 - Local-Proposed | 7 | 0.1% | 14.3% | 42.9% | 14.3% | 0.0% |
| OSHPD 5-Proposed | 4 | 0.1% | 0.0% | 25.0% | 100.0% | 0.0% |
| OSHPD 1R-Skilled Nursing Only | 3 | 0.1% | 66.7% | 100.0% | 100.0% | 100.0% |
| OSHPD 1R-Acute Psych / SNF only | 3 | 0.1% | 100.0% | 100.0% | 100.0% | 100.0% |
| OSHPD 1-No Gen Acute Care - OSHPD Bldg | 3 | 0.1% | 0.0% | 100.0% | 66.7% | 100.0% |
| OSHPD 1R-Delicensed Clinic under OSHPD | 2 | 0.0% | 50.0% | 100.0% | 100.0% | 100.0% |
| OSHPD 2-Proposed | 2 | 0.0% | 0.0% | 100.0% | 100.0% | 0.0% |
| OSHPD 5-Not an Independent Building | 2 | 0.0% | 0.0% | 0.0% | 100.0% | 100.0% |
| Adjacent Building - Local Jurisdiction-Outpatient Only | 1 | 0.0% | 100.0% | 100.0% | 100.0% | 100.0% |
| OSHPD 5-Acute Psych / SNF only | 1 | 0.0% | 0.0% | 100.0% | 100.0% | 100.0% |
| OSHPD 1R-Acute Psych Only | 1 | 0.0% | 100.0% | 100.0% | 100.0% | 100.0% |
| OSHPD 1R-Tunnels | 1 | 0.0% | 0.0% | 100.0% | 100.0% | 100.0% |
| OSHPD 2-Under Construction | 1 | 0.0% | 0.0% | 0.0% | 100.0% | 0.0% |
| Local-Tanks | 1 | 0.0% | 0.0% | 100.0% | 0.0% | 0.0% |

## Height and stories

Zero measurements need review by building status before treating them as errors. Extreme heights may be legitimate; the sample below provides context rather than an exclusion rule.


In [13]:
zero_records = [r for r in rows if number(r["Height (ft)"]) == 0 or number(r["Stories"]) == 0]
zero_status = []
for status, count in Counter(r["Building Status"] for r in zero_records).most_common():
    group = [r for r in rows if r["Building Status"] == status]
    zero_status.append([status, sum(number(r["Height (ft)"]) == 0 for r in group), sum(number(r["Stories"]) == 0 for r in group)])
display_sections([
    table(["Measurement", "Zeros", "Negative", "Fractional"], [
        [c, numeric[c]["zero"], numeric[c]["negative"], numeric[c]["fractional"]]
        for c in ("Height (ft)", "Stories")
    ]),
    "Fractional heights are expected; fractional story counts, if present, need review.",
    table(["Status with zero measurements", "Zero height", "Zero stories"], zero_status),
])


| Measurement | Zeros | Negative | Fractional |
| --- | --- | --- | --- |
| Height (ft) | 22 | 0 | 1292 |
| Stories | 48 | 0 | 0 |

Fractional heights are expected; fractional story counts, if present, need review.

| Status with zero measurements | Zero height | Zero stories |
| --- | --- | --- |
| OSHPD 1-Equipment Yard | 18 | 21 |
| OSHPD 1-In Service | 1 | 14 |
| OSHPD 1-Tunnels | 0 | 6 |
| OSHPD 1-Under Construction | 3 | 2 |
| OSHPD 1-Proposed | 0 | 2 |
| OSHPD 3 - Local-Outpatient Only | 0 | 1 |
| OSHPD 1R-Tunnels | 0 | 1 |
| OSHPD 1-Tanks | 0 | 1 |

In [14]:
tallest = sorted(
    [r for r in rows if number(r["Height (ft)"]) is not None],
    key=lambda r: number(r["Height (ft)"]), reverse=True,
)[:5]
pairs = [(number(r["Height (ft)"]), number(r["Stories"])) for r in rows]
positive_pairs = [(h, s) for h, s in pairs if h is not None and s is not None and h > 0 and s > 0]
correlation = None
if len(positive_pairs) > 1:
    heights, stories = zip(*positive_pairs)
    if len(set(heights)) > 1 and len(set(stories)) > 1:
        correlation = statistics.correlation(heights, stories)
display_sections([
    "### Five tallest records",
    table(["Building ID", "Status", "Height (ft)", "Stories"], [
        [r["Building Nbr"], r["Building Status"], r["Height (ft)"], r["Stories"] or "(blank)"] for r in tallest
    ]),
    f"Height/story Pearson correlation: **{fmt(correlation)}**, using {len(positive_pairs):,} records with both measurements positive. This describes the available subset, not a causal relationship or a data-validity rule.",
])


### Five tallest records

| Building ID | Status | Height (ft) | Stories |
| --- | --- | --- | --- |
| BLD-05878 | OSHPD 1-In Service | 262.5 | 17 |
| BLD-03801 | OSHPD 1-In Service | 199.25 | 12 |
| BLD-01013 | OSHPD 1-In Service | 195.5 | 15 |
| BLD-01012 | OSHPD 1-In Service | 195 | 15 |
| BLD-02986 | OSHPD 1-In Service | 192 | 12 |

Height/story Pearson correlation: **0.9682**, using 2,283 records with both measurements positive. This describes the available subset, not a causal relationship or a data-validity rule.

## Building codes and completion years

Preserve the code family and edition year separately. A completion year earlier than its code edition is a review flag; the dataset alone does not explain the discrepancy.


In [15]:
def code_parts(label):
    match = re.fullmatch(r"(\d{4})\s+(.+)", label.strip())
    return (int(match[1]), match[2]) if match else (None, label)

code_families = Counter(code_parts(r["Building Code"])[1] for r in rows)
known_code_rows = [r for r in rows if code_parts(r["Building Code"])[0] is not None]
code_mismatches = [
    r for r in known_code_rows
    if number(r["Building Code Year"]) is not None
    and code_parts(r["Building Code"])[0] != number(r["Building Code Year"])
]
chronology_flags = [
    r for r in rows
    if number(r["Year Completed"]) is not None and number(r["Building Code Year"]) is not None
    and number(r["Year Completed"]) < number(r["Building Code Year"])
]
future_years = {
    c: sum(v > SNAPSHOT_YEAR for v in numeric[c]["values"])
    for c in ("Building Code Year", "Year Completed")
}
display_sections([
    table(["Code family/label", "Records", "% of all"], [[v, count, rate(count, n)] for v, count in code_families.most_common()]),
    table(["Year check", "Records"], [
        ["Code label year disagrees with numeric code year (both supplied)", len(code_mismatches)],
        ["Year-prefixed code label but missing/invalid numeric code year", sum(number(r["Building Code Year"]) is None for r in known_code_rows)],
        ["Unknown code label with blank code year", sum(r["Building Code"] == "Unknown" and blank(r["Building Code Year"]) for r in rows)],
        ["Completion year precedes code year", len(chronology_flags)],
        [f"Code year after {SNAPSHOT_YEAR}", future_years["Building Code Year"]],
        [f"Completion year after {SNAPSHOT_YEAR}", future_years["Year Completed"]],
        ["Fractional code year", numeric["Building Code Year"]["fractional"]],
        ["Fractional completion year", numeric["Year Completed"]["fractional"]],
    ]),
])


| Code family/label | Records | % of all |
| --- | --- | --- |
| California Building Code (CBC) | 3685 | 78.6% |
| Uniform Building Code (UBC) | 831 | 17.7% |
| Unknown | 80 | 1.7% |
| City of Los Angeles (COLA) | 68 | 1.4% |
| County of Los Angeles (LAC) | 24 | 0.5% |
| California Admin Code (CAC) | 2 | 0.0% |

| Year check | Records |
| --- | --- |
| Code label year disagrees with numeric code year (both supplied) | 0 |
| Year-prefixed code label but missing/invalid numeric code year | 0 |
| Unknown code label with blank code year | 80 |
| Completion year precedes code year | 3 |
| Code year after 2026 | 0 |
| Completion year after 2026 | 0 |
| Fractional code year | 0 |
| Fractional completion year | 0 |

In [16]:
display_sections([
    f"### Completion/code chronology flags ({len(chronology_flags)} total; up to 10 shown)",
    table(["Building ID", "Status", "Code label", "Code year", "Completed"], [
        [r["Building Nbr"], r["Building Status"], r["Building Code"], r["Building Code Year"], r["Year Completed"]]
        for r in chronology_flags[:10]
    ]),
])
decades = Counter(int(v) // 10 * 10 for v in numeric["Year Completed"]["values"] if v.is_integer())
in_service_decades = Counter(
    int(v) // 10 * 10 for r in in_service
    if (v := number(r["Year Completed"])) is not None and v.is_integer()
)
display_sections([
    "### Completion decade",
    table(["Decade", "All records", "OSHPD 1-In Service"], [[f"{d}s", count, in_service_decades[d]] for d, count in sorted(decades.items())]),
    "Decades include only supplied whole-number years. Missing completion years should not be inferred from code editions.",
])


### Completion/code chronology flags (3 total; up to 10 shown)

| Building ID | Status | Code label | Code year | Completed |
| --- | --- | --- | --- | --- |
| BLD-01909 | OSHPD 1-In Service | 1985 California Building Code (CBC) | 1985 | 1978 |
| BLD-07237 | OSHPD 1-In Service | 1998 California Building Code (CBC) | 1998 | 1997 |
| BLD-07216 | OSHPD 1-In Service | 2010 California Building Code (CBC) | 2010 | 1995 |

### Completion decade

| Decade | All records | OSHPD 1-In Service |
| --- | --- | --- |
| 1900s | 2 | 2 |
| 1910s | 3 | 0 |
| 1920s | 17 | 8 |
| 1930s | 14 | 8 |
| 1940s | 36 | 27 |
| 1950s | 209 | 165 |
| 1960s | 402 | 304 |
| 1970s | 524 | 460 |
| 1980s | 473 | 420 |
| 1990s | 716 | 623 |
| 2000s | 471 | 412 |
| 2010s | 662 | 553 |
| 2020s | 272 | 187 |

Decades include only supplied whole-number years. Missing completion years should not be inferred from code editions.

## SPC ratings and notices

Compare structural ratings across the full snapshot and the in-service subset. Keep suffixes intact. Ratings and notices describe seismic assessments and need leakage review before use as predictors.


In [17]:
subcounts = Counter(r["SPC Rating *"] for r in in_service)
display_sections([
    table(["SPC rating", "All records", "% of all", "OSHPD 1-In Service", "% of subset"], [
        [v or "(blank)", count, rate(count, n), subcounts[v], rate(subcounts[v], len(in_service))]
        for v, count in counts["SPC Rating *"].most_common()
    ]),
    table(["AB 1882 Notice", "Records", "% of all"], [
        [v or "(blank)", count, rate(count, n)] for v, count in counts["AB 1882 Notice"].most_common()
    ]),
])


| SPC rating | All records | % of all | OSHPD 1-In Service | % of subset |
| --- | --- | --- | --- | --- |
| 5 | 1251 | 26.7% | 1242 | 39.0% |
| N/A | 1141 | 24.3% | 1 | 0.0% |
| 4 | 718 | 15.3% | 717 | 22.5% |
| 2 | 636 | 13.6% | 634 | 19.9% |
| 5s | 451 | 9.6% | 97 | 3.0% |
| 3 | 333 | 7.1% | 333 | 10.5% |
| 4s | 61 | 1.3% | 61 | 1.9% |
| 3s | 46 | 1.0% | 46 | 1.4% |
| 4D | 29 | 0.6% | 29 | 0.9% |
| 1 | 17 | 0.4% | 17 | 0.5% |
| 2s | 6 | 0.1% | 5 | 0.2% |
| 1s | 1 | 0.0% | 1 | 0.0% |

| AB 1882 Notice | Records | % of all |
| --- | --- | --- |
| (blank) | 3936 | 83.9% |
| This building does not significantly jeopardize life, but may not be repairable or functional following an earthquake. | 636 | 13.6% |
| Earthquake Resilient | 118 | 2.5% |

## Identifiers, locations, and URLs

Shared facility names or coordinates do not justify deduplication. Coordinates are checked against global bounds only. URL checks assess syntax and hostnames, not whether the pages are reachable.


In [18]:
parsed_urls = [urlparse(r["Building URL"]) for r in rows]
url_hosts = Counter(url.netloc for url in parsed_urls)
invalid_urls = sum(url.scheme not in {"http", "https"} or not url.netloc for url in parsed_urls)
checks = [
    ["Blank facility IDs", sum(blank(r["Perm ID"]) for r in rows)],
    ["Blank building IDs", sum(blank(r["Building Nbr"]) for r in rows)],
    ["Facility IDs not matching five digits", sum(re.fullmatch(r"\d{5}", r["Perm ID"]) is None for r in rows)],
    ["Building IDs not matching BLD- plus five digits", sum(re.fullmatch(r"BLD-\d{5}", r["Building Nbr"]) is None for r in rows)],
    ["Exact duplicate rows", duplicate_rows],
    ["Duplicate facility/building key pairs", duplicate_keys],
    ["Repeated building IDs", n - len(counts["Building Nbr"])],
    ["Facility names shared by multiple facility IDs", sum(len(v) > 1 for v in names.values())],
    ["Unique coordinate pairs", len(coordinates)],
    ["Coordinate pairs shared by multiple facility IDs", sum(len(v) > 1 for v in coordinates.values())],
    ["Missing, invalid, or out-of-global-range latitude", sum(number(r["Latitude"]) is None or not -90 <= number(r["Latitude"]) <= 90 for r in rows)],
    ["Missing, invalid, or out-of-global-range longitude", sum(number(r["Longitude"]) is None or not -180 <= number(r["Longitude"]) <= 180 for r in rows)],
    ["Unique building URLs", len(counts["Building URL"])],
    ["Missing or invalid HTTP(S) URLs", invalid_urls],
    ["Cells with surrounding whitespace", sum(v != v.strip() for r in rows for v in r.values())],
]
display_sections([
    table(["Check", "Count"], checks),
    table(["URL host", "Records"], url_hosts.most_common()),
])


| Check | Count |
| --- | --- |
| Blank facility IDs | 0 |
| Blank building IDs | 0 |
| Facility IDs not matching five digits | 0 |
| Building IDs not matching BLD- plus five digits | 0 |
| Exact duplicate rows | 0 |
| Duplicate facility/building key pairs | 0 |
| Repeated building IDs | 0 |
| Facility names shared by multiple facility IDs | 7 |
| Unique coordinate pairs | 422 |
| Coordinate pairs shared by multiple facility IDs | 1 |
| Missing, invalid, or out-of-global-range latitude | 0 |
| Missing, invalid, or out-of-global-range longitude | 0 |
| Unique building URLs | 4690 |
| Missing or invalid HTTP(S) URLs | 0 |
| Cells with surrounding whitespace | 0 |

| URL host | Records |
| --- | --- |
| esp.oshpd.ca.gov | 4690 |

In [19]:
display_sections([
    "### Facility-level consistency",
    table(["Field", "Facility IDs with multiple values"], [
        [c, sum(len({r[c] for r in group}) > 1 for group in facilities.values())]
        for c in ["Facility Name", "County Code", "City", "Latitude", "Longitude"]
    ]),
    table(["Facility ID", "City labels and record counts"], [
        [key, "; ".join(f"{city}: {count}" for city, count in values.most_common())]
        for key, values in sorted(city_conflicts.items())
    ]) if city_conflicts else "No within-facility city conflicts found.",
])


### Facility-level consistency

| Field | Facility IDs with multiple values |
| --- | --- |
| Facility Name | 0 |
| County Code | 0 |
| City | 2 |
| Latitude | 0 |
| Longitude | 0 |

| Facility ID | City labels and record counts |
| --- | --- |
| 11000 | Fall River Mills: 9; Burney: 1 |
| 18219 | Redwood City: 2; Redwoord City: 1 |

In [20]:
display_sections([
    "### Ten counties with the most records",
    table(["County", "Building records", "% of all", "Facility IDs"], [
        [county, count, rate(count, n), len({r["Perm ID"] for r in rows if r["County Code"] == county})]
        for county, count in counts["County Code"].most_common(10)
    ]),
    "County counts show dataset representation, not earthquake risk.",
])


### Ten counties with the most records

| County | Building records | % of all | Facility IDs |
| --- | --- | --- | --- |
| 19 - Los Angeles | 1040 | 22.2% | 91 |
| 30 - Orange | 383 | 8.2% | 36 |
| 37 - San Diego | 367 | 7.8% | 26 |
| 36 - San Bernardino | 307 | 6.5% | 25 |
| 33 - Riverside | 219 | 4.7% | 20 |
| 43 - Santa Clara | 218 | 4.6% | 17 |
| 01 - Alameda | 161 | 3.4% | 16 |
| 34 - Sacramento | 145 | 3.1% | 14 |
| 10 - Fresno | 141 | 3.0% | 10 |
| 39 - San Joaquin | 139 | 3.0% | 8 |

County counts show dataset representation, not earthquake risk.

## Cleanup decisions

Prioritize mechanical loading fixes, source questions, and population decisions. Findings below are review candidates; none are automatically corrected or removed.


In [21]:
issues = [
    ["Loading", "Encoding and headers", "cp1252; source header is SPC Rating *", "Keep raw files unchanged; normalize headers in analysis. Map the SPC header explicitly when joining notebook 01's dataset."],
    ["Typing", "Identifiers and markers", "Numeric-looking Perm ID; blanks, Unknown, and N/A have different meanings", "Keep IDs as text and preserve missing reasons; parse measurement columns as nullable numbers."],
    ["Review", "Zero measurements", f"{numeric['Height (ft)']['zero']} zero heights; {numeric['Stories']['zero']} zero story counts", "Review the building status and source before converting zeros to missing or excluding records."],
    ["Review", "Code/completion chronology", f"{len(chronology_flags)} completion years precede code years", "Check original building records; do not swap years or infer a correction."],
    ["Review", "City labels", f"{len(city_conflicts)} facility IDs have multiple city labels", "Confirm spelling errors separately from potentially legitimate location differences."],
    ["Population", "Mixed record types", f"{len(counts['Building Status'])} statuses; {n-len(in_service):,} outside the exact in-service subset", "Document inclusion rules and record counts before modeling."],
    ["Features", "Incomplete physical characteristics", f"{len(complete_features):,}/{n:,} records have all four numeric features, including zeros", "Assess missingness by status; do not adopt complete-case filtering or imputation without a documented rationale."],
    ["Features", "Building age versus code edition", f"{counts['Building Code']['Unknown']} Unknown code labels", "Retain code family and edition; derive age only from validated completion years and an explicit reference year."],
    ["Features", "Assessment leakage and identifiers", "SPC, AB 1882 Notice, URLs, and IDs", "Separate candidate building predictors from assessment targets; use IDs/URLs for linkage, and review facility-grouped evaluation."],
    ["Features", "Count helper", f"Observed values: {dict(counts['Count'])}", "If constant, exclude from predictors."],
]
display_sections([table(["Action", "Issue", "Evidence", "Recommended handling"], issues)])
assert SOURCE.read_bytes() == raw, "The input changed during analysis; refresh all results."


| Action | Issue | Evidence | Recommended handling |
| --- | --- | --- | --- |
| Loading | Encoding and headers | cp1252; source header is SPC Rating * | Keep raw files unchanged; normalize headers in analysis. Map the SPC header explicitly when joining notebook 01's dataset. |
| Typing | Identifiers and markers | Numeric-looking Perm ID; blanks, Unknown, and N/A have different meanings | Keep IDs as text and preserve missing reasons; parse measurement columns as nullable numbers. |
| Review | Zero measurements | 22 zero heights; 48 zero story counts | Review the building status and source before converting zeros to missing or excluding records. |
| Review | Code/completion chronology | 3 completion years precede code years | Check original building records; do not swap years or infer a correction. |
| Review | City labels | 2 facility IDs have multiple city labels | Confirm spelling errors separately from potentially legitimate location differences. |
| Population | Mixed record types | 28 statuses; 1,507 outside the exact in-service subset | Document inclusion rules and record counts before modeling. |
| Features | Incomplete physical characteristics | 2,289/4,690 records have all four numeric features, including zeros | Assess missingness by status; do not adopt complete-case filtering or imputation without a documented rationale. |
| Features | Building age versus code edition | 80 Unknown code labels | Retain code family and edition; derive age only from validated completion years and an explicit reference year. |
| Features | Assessment leakage and identifiers | SPC, AB 1882 Notice, URLs, and IDs | Separate candidate building predictors from assessment targets; use IDs/URLs for linkage, and review facility-grouped evaluation. |
| Features | Count helper | Observed values: {'1': 4690} | If constant, exclude from predictors. |